-qq: quiet mode (hiển thị log khi có lỗi)

In [1]:
!pip install -qq faiss-cpu
!pip install -qq transformers
!pip install -qq pandas
!pip install -qq numpy
!pip install -qq scikit-learn
!pip install -qq tqdm

import thư viện

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F # hàm activation function
import faiss # hàm tìm kiếm vector
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm # hiển thị progress bar
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

Đọc dữ liệu

In [3]:
df = pd.read_csv('./cls_spam_text.csv')
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


Phân chia dữ liệu

In [4]:
messages = df['Message'].values.tolist()
labels = df['Category'].values.tolist()
labels[0]

'ham'

Chuẩn bị mô hình embedding

In [ ]:
# Load embedding model
MODEL_NAME = 'intfloat/multilingual-e5-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) 
model = AutoModel.from_pretrained(MODEL_NAME)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval() # để chạy model trong chế độ kiểm tra

print(f"Using device: {device}")
print(f"Model loaded: {MODEL_NAME}")

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

c:\Users\HP\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--intfloat--multilingual-e5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. 

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Using device: cuda
Model loaded: intfloat/multilingual-e5-base


Chuẩn bị mô hình embedding

In [ ]:
def average_pooling(last_hidden_states, attention_mask):
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[:, None]


In [ ]:
# Hàm encode
def encode_text(batch_texts,model, tokenizer, device):
    # Tokenize input
    encoded_input = tokenizer(batch_texts,
                              max_length=512, 
                              padding=True, 
                              truncation=True, 
                              return_tensors='pt').to(device)
    
    # Move to device (đưa tới GPU)
    encoded_input = {k: v.to(device) for k,v in encoded_input.items()}
    
    # Generate embeddings
    with torch.no_grad():
        outputs = model(**encoded_input)
        batch_embeddings = average_pooling(outputs.last_hidden_state, encoded_input['attention_mask'])
        
        # Normalize embeddings
        batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)
        
    return batch_embeddings.cpu().numpy()

Vector hoá dữ liệu

In [ ]:
# Hàm tạo sentence embedding
def create_sentence_embeddings(texts, model, tokenizer, device, batch_size=32):
    """Tạo embedding cho các câu văn bản"""
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding texts"):
        # 0 là bắt đầu, len(texts) là kết thúc, batch_size là số lượng mỗi lần
        batch_texts = texts[i:i+batch_size]
        batch_embeddings = encode_text(batch_texts, model, tokenizer, device)
        embeddings.append(batch_embeddings)
        
    return np.vstack(embeddings) # vstack là ghép các vector embedding theo chiều dọc


In [10]:
# Chuẩn bị labels
le = LabelEncoder()
y = le.fit_transform(labels)

# Tạo embedding cho dữ liệu
X_embeddings = create_sentence_embeddings(messages, model, tokenizer, device)

# Tạo metadata
metadata = []
for i, (message, label) in enumerate(zip(messages, labels)):
    metadata.append({
                    "index": i,
                    "message": message, 
                    "label": label, 
                    "label_encoded": y[i]
                    })

Encoding texts: 100%|██████████| 175/175 [04:02<00:00,  1.39s/it]


In [11]:
# Split data into train and test (90% train, 10% test)
TEST_SIZE = 0.1
SEED = 42

train_indices, test_indices = train_test_split(
    range(len(messages)),
    test_size=TEST_SIZE,
    stratify=y,
    random_state=SEED
)

In [12]:
# Split embeddings and metadata
X_train_emb = X_embeddings[train_indices]
X_test_emb = X_embeddings[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]

train_metadata = [metadata[i] for i in train_indices]
test_metadata = [metadata[i] for i in test_indices]

print(f"Train size: {len(X_train_emb)}")
print(f"Test size: {len(X_test_emb)}")
print(f"Train label distribution: {np.bincount(y_train)}")
print(f"Test label distribution: {np.bincount(y_test)}")

Train size: 5014
Test size: 558
Train label distribution: [4342  672]
Test label distribution: [483  75]


In [ ]:
# Create FAISS index
embedding_dim = X_train_emb.shape[1]

index = faiss.IndexFlatIP(embedding_dim) # tạo hộp chứa có số chiều embedding
index.add(X_train_emb.astype('float32')) # thêm vector embedding vào hộp

print(f"FAISS index created with {index.ntotal} vectors")

FAISS index created with 5014 vectors


In [ ]:
def classify_with_knn(text, model, tokenizer, device, index, metadata, k=3):

    # Tokenize text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # Average pooling
    last_hidden = outputs.last_hidden_state
    attention_mask = inputs["attention_mask"]

    last_hidden = last_hidden.masked_fill(~attention_mask[..., None].bool(), 0.0)
    embedding = last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

    # Normalize
    embedding = F.normalize(embedding, p=2, dim=1)

    query_vector = embedding.cpu().numpy().astype("float32")

    # Search FAISS
    scores, indices = index.search(query_vector, k)
    # indices là index của vector embedding trong hộp

    neighbors = []
    for score, idx in zip(scores[0], indices[0]):
        item = metadata[idx]
        neighbors.append({
            "label": item["label"],
            "message": item["message"],
            "score": float(score)
        })

    # Majority vote
    labels = [n["label"] for n in neighbors]
    prediction = max(set(labels), key=labels.count)
    # key=labels.count là tìm phần tử xuất hiện nhiều nhất trong labels

    return prediction, neighbors

In [18]:
def spam_classifier_pipeline(user_input, k=3):
    """
    Complete pipeline for spam classification

    Args:
        user_input (str): Text to classify
        k (int): Number of nearest neighbors to consider

    Returns:
        dict: Classification results with details
    """

    print()
    print(f"***Classifying: '{user_input}'")
    print()
    print(f"***Using top-{k} nearest neighbors")
    print()

    # Get prediction and neighbors
    prediction, neighbors = classify_with_knn(
        user_input, model, tokenizer, device, index, train_metadata, k=k
    )

    # Display results
    print(f"***Prediction: {prediction.upper()}")
    print()

    print("***Top neighbors:")
    for i, neighbor in enumerate(neighbors, 1):
        print(f"{i}. Label: {neighbor['label']} | Score: {neighbor['score']:.4f}")
        print(f"   Message: {neighbor['message']}")
        print()

    # Count label distribution
    labels = [n['label'] for n in neighbors]
    label_counts = {label: labels.count(label) for label in set(labels)}

    return {
        'prediction': prediction,
        'neighbors': neighbors,
        'label_distribution': label_counts
    }

In [19]:

# Test với các ví dụ khác nhau
test_examples = [
    "I am actually thinking a way of doing something useful",
    "FREE!! Click here to win $1000 NOW! Limited time offer!",
    "Hey, can you pick me up at 5pm today?",
    "URGENT: Your account will be suspended unless you verify your details NOW",
    "Thanks for the meeting today, let's schedule the next one for next week",
    "Congratulations! You've won a prize! Call this number to claim it"
]

print("Testing pipeline with different examples:")

for i, example in enumerate(test_examples, 1):
    print(f"\n***Example {i}:")
    result = spam_classifier_pipeline(example, k=3)

Testing pipeline with different examples:

***Example 1:

***Classifying: 'I am actually thinking a way of doing something useful'

***Using top-3 nearest neighbors

***Prediction: HAM

***Top neighbors:
1. Label: ham | Score: 0.8503
   Message: You're right I have now that I think about it

2. Label: ham | Score: 0.8493
   Message: K, I'll work something out

3. Label: ham | Score: 0.8478
   Message: I have gone into get info bt dont know what to do


***Example 2:

***Classifying: 'FREE!! Click here to win $1000 NOW! Limited time offer!'

***Using top-3 nearest neighbors

***Prediction: SPAM

***Top neighbors:
1. Label: spam | Score: 0.8884
   Message: FREE MESSAGE Activate your 500 FREE Text Messages by replying to this message with the word FREE For terms & conditions, visit www.07781482378.com

2. Label: spam | Score: 0.8884
   Message: FREE MESSAGE Activate your 500 FREE Text Messages by replying to this message with the word FREE For terms & conditions, visit www.07781482378.com